# Participant/session-level leakage verification for the tooth-segmentation dataset (DATASET DIENTES) — addresses the co-advisor's Correc2 comment on Section 2.2.1/2.3.1. Checks (1) train/valid/test of the segmenter share no participant identifier, and (2) participants used to train/validate the segmenter are absent from the final integrated-pipeline evaluation set (Dataset Tratado 3 test).

Context (do not delete): the co-advisor noted that Section 2.2.1 reports the
1,833 segmentation images split 1,650/110/73, and Section 2.3.1 only mentions
a "structural split" — it never states whether all photographs from the same
participant/session stayed inside a single partition. This matters because
the underlying Ahmed et al. dataset has up to ten photographs per participant
(five with retractor, five without, across multiple views). This notebook
runs the two checks she asked for, using ONLY image filenames already present
on disk — no re-annotation, no retraining, no change to any existing split.

If both results are 0 matches, add to the end of the first paragraph of
Section 2.2.1 (verbatim, as instructed):

> The segmentation dataset split was additionally verified at
> participant/session level, confirming that no identifier was shared among
> the training, validation, and test subsets. A cross-stage identifier-level
> check also confirmed that participants represented in the segmentation
> training/validation subsets were not included in the final evaluation
> subset of the integrated pipeline.

If NOT both zero: do **not** add that sentence. Instead report the actual
counts to the co-advisor and treat the affected participants as a limitation
(Section 4) or re-split DATASET DIENTES by group before re-running the
`pipeline_YOLO26{n,s,m}_dientes.md` notebooks.

## 0. Setup


In [1]:
import re
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
BASE_CARIES = Path("/content/drive/MyDrive/UNIVERSIDAD/9NO SEMESTRE/TITULACION/DATASET CARIES")
BASE_DIENTES = Path("/content/drive/MyDrive/UNIVERSIDAD/9NO SEMESTRE/TITULACION/DATASET DIENTES")

# Final integrated-pipeline evaluation set (test split used in
# pipeline_asociacion_geometrica.md, Section 3.5/3.6 of the article).
DATASET_TRATADO_3 = BASE_CARIES / "Dataset Tratado 3"

assert BASE_DIENTES.exists(), f"No se encontró {BASE_DIENTES}"
assert DATASET_TRATADO_3.exists(), f"No se encontró {DATASET_TRATADO_3}"

OUTPUT_DIR = BASE_CARIES  # guardamos las tablas de evidencia junto a los CSV ya existentes

## 1. Extracting the participant/session identifier from DATASET DIENTES filenames

The images in `DATASET DIENTES` come from the Roboflow-style export used in
`pipeline_YOLO26n/s/m_dientes.md`, with filenames such as:

```
c_Mandibular_anonymous_003-007-1214-00_1732862917124_Mandibular_View_jpg.rf.<hash>.jpg
c_Maxillary_Occlusal_anonymous-maxillaryView-1726650785522_jpg.rf.<hash>.jpg
```

The identifier block (`003-007-1214-00`, or the raw pilot filename when no
block-style ID exists) is captured with the same pattern already validated in
`Tratado_2_dataset_Ahmed.md`, so results stay consistent with the rest of the
thesis.


In [3]:
PATRON_ID = re.compile(r"(\d{3}[-_]\d{3}[-_]\d+[-_]\d{2})")

def extraer_grupo(stem: str) -> str:
    m = PATRON_ID.search(stem)
    if m:
        return re.sub(r"[-_]", "-", m.group(1))  # normaliza separador
    # Fallback: pilot-style filenames without the block pattern
    # (e.g. anonymous-maxillaryView-1726650785522) -> usa el bloque numérico
    # más largo como proxy de sesión; si no hay ninguno, la imagen es su propio grupo.
    m2 = re.compile(r"(\d{10,})").search(stem)  # timestamp largo -> mismo criterio que Tratado_2
    if m2:
        return m2.group(1)
    return stem

IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

registros_dientes = []
for split in ["train", "valid", "test"]:
    img_dir = BASE_DIENTES / split / "images"
    for img_p in sorted(img_dir.glob("*")):
        if img_p.suffix.lower() not in IMG_EXTENSIONS:
            continue
        registros_dientes.append({
            "split": split,
            "stem": img_p.stem,
            "path": img_p,
            "grupo": extraer_grupo(img_p.stem),
        })

print(f"Total de imágenes indexadas en DATASET DIENTES: {len(registros_dientes)}")
for split in ["train", "valid", "test"]:
    n = sum(1 for r in registros_dientes if r["split"] == split)
    print(f"  {split}: {n} imágenes")

Total de imágenes indexadas en DATASET DIENTES: 1833
  train: 1650 imágenes
  valid: 110 imágenes
  test: 73 imágenes


## 2. CHECK 1 — Group overlap across train / valid / test of the segmenter


In [4]:
grupos_train = {r["grupo"] for r in registros_dientes if r["split"] == "train"}
grupos_valid = {r["grupo"] for r in registros_dientes if r["split"] == "valid"}
grupos_test = {r["grupo"] for r in registros_dientes if r["split"] == "test"}

interseccion_train_valid = grupos_train & grupos_valid
interseccion_train_test = grupos_train & grupos_test
interseccion_valid_test = grupos_valid & grupos_test

tabla_check1 = pd.DataFrame({
    "Comprobación": [
        "Identificadores en train",
        "Identificadores en valid",
        "Identificadores en test",
        "Coincidencias train–valid",
        "Coincidencias train–test",
        "Coincidencias valid–test",
    ],
    "Resultado": [
        len(grupos_train), len(grupos_valid), len(grupos_test),
        len(interseccion_train_valid), len(interseccion_train_test), len(interseccion_valid_test),
    ],
})
print(tabla_check1.to_string(index=False))

RUTA_CHECK1 = OUTPUT_DIR / "tabla_evidencia_particion_dientes.csv"
tabla_check1.to_csv(RUTA_CHECK1, index=False)
print(f"\nGuardada en: {RUTA_CHECK1}")

if interseccion_train_valid or interseccion_train_test or interseccion_valid_test:
    print("\n⚠️ HAY FUGA por participante/sesión en DATASET DIENTES. Identificadores compartidos:")
    print(" train∩valid:", sorted(interseccion_train_valid)[:20])
    print(" train∩test :", sorted(interseccion_train_test)[:20])
    print(" valid∩test :", sorted(interseccion_valid_test)[:20])
else:
    print("\n✅ Cero coincidencias entre particiones del segmentador confirmado.")

             Comprobación  Resultado
 Identificadores en train        990
 Identificadores en valid        109
  Identificadores en test         72
Coincidencias train–valid         13
 Coincidencias train–test          8
 Coincidencias valid–test          1

Guardada en: /content/drive/MyDrive/UNIVERSIDAD/9NO SEMESTRE/TITULACION/DATASET CARIES/tabla_evidencia_particion_dientes.csv

⚠️ HAY FUGA por participante/sesión en DATASET DIENTES. Identificadores compartidos:
 train∩valid: ['003-007-1097-01', '003-007-1230-01', '003-007-1300-01', '003-007-1387-01', '003-007-1424-01', '003-007-1467-01', '003-007-668-01', '003-007-672-01', '003-007-773-01', '003-008-1185-01', '003-008-1203-01', '003-008-748-01', '004-008-916-01']
 train∩test : ['003-007-1146-01', '003-007-1427-01', '003-007-471-01', '003-007-610-01', '003-007-953-01', '003-008-1183-01', '003-103-435-01', '005-007-1023-01']
 valid∩test : ['003-103-430-01']


## 3. CHECK 2 — Cross-stage: segmenter train/valid participants vs. final integrated-pipeline evaluation set

The final evaluation of the integrated pipeline (`pipeline_asociacion_geometrica.md`,
Tables 16–18) runs on the **test** split of `Dataset Tratado 3` (caries
dataset). This check confirms that no participant used to *train or validate*
the segmenter reappears there — a stricter, identifier-level complement to
the SHA-256/pHash check already done in `Tratado_0_Deduplicacion_Cruzada.md`
and `Tratado_2_dataset_Ahmed.md` (Cell 7.1), which only catch exact/near-duplicate
images, not different photographs of the same participant.


In [5]:
PATRON_ID_CARIES = re.compile(r"(\d{3}[-_]\d{3}[-_]\d+[-_]\d{2})")

def extraer_grupo_caries(stem: str) -> str:
    m = PATRON_ID_CARIES.search(stem)
    if m:
        return re.sub(r"[-_]", "-", m.group(1))
    return stem  # imagen huérfana / externa (p.ej. healthy_*) -> grupo = ella misma

grupos_dientes_train_valid = grupos_train | grupos_valid  # lo que el segmentador "vio"

grupos_evaluacion_final = set()
for img_p in (DATASET_TRATADO_3 / "images" / "test").glob("*"):
    grupos_evaluacion_final.add(extraer_grupo_caries(img_p.stem))

coincidencias_cross_stage = grupos_dientes_train_valid & grupos_evaluacion_final

tabla_check2 = pd.DataFrame({
    "Comprobación": [
        "Identificadores en segmentador (train+valid)",
        "Identificadores en evaluación final del pipeline (test)",
        "Coincidencias segmentador(train+valid) – evaluación final",
    ],
    "Resultado": [
        len(grupos_dientes_train_valid),
        len(grupos_evaluacion_final),
        len(coincidencias_cross_stage),
    ],
})
print(tabla_check2.to_string(index=False))

RUTA_CHECK2 = OUTPUT_DIR / "tabla_evidencia_cross_stage_dientes_vs_evaluacion.csv"
tabla_check2.to_csv(RUTA_CHECK2, index=False)
print(f"\nGuardada en: {RUTA_CHECK2}")

if coincidencias_cross_stage:
    print(f"\n⚠️ {len(coincidencias_cross_stage)} participante(s) del segmentador (train/valid) "
          "reaparecen en la evaluación final del pipeline:")
    print(sorted(coincidencias_cross_stage)[:30])
else:
    print("\n✅ Cero coincidencias cross-stage confirmado: ningún participante del "
          "entrenamiento/validación del segmentador aparece en la evaluación final del pipeline.")

                                             Comprobación  Resultado
             Identificadores en segmentador (train+valid)       1086
  Identificadores en evaluación final del pipeline (test)        190
Coincidencias segmentador(train+valid) – evaluación final         26

Guardada en: /content/drive/MyDrive/UNIVERSIDAD/9NO SEMESTRE/TITULACION/DATASET CARIES/tabla_evidencia_cross_stage_dientes_vs_evaluacion.csv

⚠️ 26 participante(s) del segmentador (train/valid) reaparecen en la evaluación final del pipeline:
['003-007-1254-01', '003-007-475-01', '003-007-610-01', '003-007-629-01', '003-007-672-01', '003-007-676-01', '003-007-787-01', '003-007-845-01', '003-007-867-01', '003-007-896-01', '003-007-898-01', '003-007-963-01', '003-008-1322-01', '003-008-495-01', '003-008-517-01', '003-008-539-01', '003-008-562-01', '003-008-622-01', '003-008-626-01', '003-008-730-01', '003-008-746-01', '003-103-299-01', '003-103-357-01', '003-103-416-01', '003-103-435-01', '004-008-914-01']


## 4. Markdown-ready evidence tables (for the response to the co-advisor / appendix)


In [6]:
print(tabla_check1.to_markdown(index=False))
print()
print(tabla_check2.to_markdown(index=False))

| Comprobación              |   Resultado |
|:--------------------------|------------:|
| Identificadores en train  |         990 |
| Identificadores en valid  |         109 |
| Identificadores en test   |          72 |
| Coincidencias train–valid |          13 |
| Coincidencias train–test  |           8 |
| Coincidencias valid–test  |           1 |

| Comprobación                                              |   Resultado |
|:----------------------------------------------------------|------------:|
| Identificadores en segmentador (train+valid)              |        1086 |
| Identificadores en evaluación final del pipeline (test)   |         190 |
| Coincidencias segmentador(train+valid) – evaluación final |          26 |


In [7]:
# ============================================================
# CHECK FINAL — Identificadores únicos afectados
# ============================================================

# Coincidencias internas dentro del dataset de segmentación
ids_train_valid = grupos_train & grupos_valid
ids_train_test = grupos_train & grupos_test
ids_valid_test = grupos_valid & grupos_test

ids_fuga_interna = (
    ids_train_valid
    | ids_train_test
    | ids_valid_test
)

# Coincidencias entre el segmentador y el dataset final de caries
ids_fuga_cross_stage = (
    grupos_dientes_train_valid
    & grupos_evaluacion_final
)

# Unión total: evita contar dos veces un mismo identificador
ids_afectados_total = (
    ids_fuga_interna
    | ids_fuga_cross_stage
)

tabla_total_afectados = pd.DataFrame({
    "Comprobación": [
        "Coincidencias train–valid",
        "Coincidencias train–test",
        "Coincidencias valid–test",
        "Identificadores únicos afectados dentro del segmentador",
        "Coincidencias segmentador–dataset final de caries",
        "Identificadores únicos afectados en total",
    ],
    "Resultado": [
        len(ids_train_valid),
        len(ids_train_test),
        len(ids_valid_test),
        len(ids_fuga_interna),
        len(ids_fuga_cross_stage),
        len(ids_afectados_total),
    ],
})

print(tabla_total_afectados.to_string(index=False))

print("\nIdentificadores afectados dentro del segmentador:")
print(sorted(ids_fuga_interna))

print("\nIdentificadores coincidentes con el dataset final de caries:")
print(sorted(ids_fuga_cross_stage))

print("\nIdentificadores afectados en total, sin duplicar:")
print(sorted(ids_afectados_total))

RUTA_TOTAL_AFECTADOS = (
    OUTPUT_DIR / "tabla_evidencia_total_identificadores_afectados.csv"
)

tabla_total_afectados.to_csv(RUTA_TOTAL_AFECTADOS, index=False)

print(f"\nTabla guardada en: {RUTA_TOTAL_AFECTADOS}")

                                           Comprobación  Resultado
                              Coincidencias train–valid         13
                               Coincidencias train–test          8
                               Coincidencias valid–test          1
Identificadores únicos afectados dentro del segmentador         22
      Coincidencias segmentador–dataset final de caries         26
              Identificadores únicos afectados en total         45

Identificadores afectados dentro del segmentador:
['003-007-1097-01', '003-007-1146-01', '003-007-1230-01', '003-007-1300-01', '003-007-1387-01', '003-007-1424-01', '003-007-1427-01', '003-007-1467-01', '003-007-471-01', '003-007-610-01', '003-007-668-01', '003-007-672-01', '003-007-773-01', '003-007-953-01', '003-008-1183-01', '003-008-1185-01', '003-008-1203-01', '003-008-748-01', '003-103-430-01', '003-103-435-01', '004-008-916-01', '005-007-1023-01']

Identificadores coincidentes con el dataset final de caries:
['003-007

## 5. Verdict — what to paste into Section 2.2.1

Only run this after Sections 2 and 3 both print ✅ with zero matches.


In [8]:
fuga_interna = bool(interseccion_train_valid or interseccion_train_test or interseccion_valid_test)
fuga_cross_stage = bool(coincidencias_cross_stage)

if not fuga_interna and not fuga_cross_stage:
    print("✅ AMBOS CHECKS EN CERO. Añadir al final del primer párrafo de la Sección 2.2.1:\n")
    print(
        "The segmentation dataset split was additionally verified at "
        "participant/session level, confirming that no identifier was shared "
        "among the training, validation, and test subsets. A cross-stage "
        "identifier-level check also confirmed that participants represented "
        "in the segmentation training/validation subsets were not included in "
        "the final evaluation subset of the integrated pipeline."
    )
else:
    print("⚠️ NO añadir la oración a 2.2.1 todavía. Hay fuga pendiente de resolver:")
    print(f"  Fuga interna (train/valid/test del segmentador): {fuga_interna}")
    print(f"  Fuga cross-stage (segmentador -> evaluación final): {fuga_cross_stage}")
    print("Reportar los conteos reales a la cotutora y considerar:")
    print(" 1) Re-particionar DATASET DIENTES por grupo (GroupShuffleSplit, igual que Tratado_2).")
    print(" 2) Si la fuga es cross-stage, excluir esas imágenes del test final o documentarlo "
          "como limitación en la Sección 4.")

⚠️ NO añadir la oración a 2.2.1 todavía. Hay fuga pendiente de resolver:
  Fuga interna (train/valid/test del segmentador): True
  Fuga cross-stage (segmentador -> evaluación final): True
Reportar los conteos reales a la cotutora y considerar:
 1) Re-particionar DATASET DIENTES por grupo (GroupShuffleSplit, igual que Tratado_2).
 2) Si la fuga es cross-stage, excluir esas imágenes del test final o documentarlo como limitación en la Sección 4.
